# Manuscript iteration diff + cohort-scoped QC

Steps 2 and 3 of the manuscript iterative-build protocol (`.cowork/skills/manuscript-iterative-build/SKILL.md`).

Point this at a manuscript cohort table and its previous iteration; it reports what changed between iterations and runs the critical QC assertions scoped to that cohort. Run it on **every** rebuild before numbers are locked.

1. **Iteration diff** — patients added/dropped, and which numeric/boolean columns moved.
2. **Cohort-scoped QC** — the concrete critical checks (SURG01, LN01–LN03, REC02/03, HIST01) filtered to this cohort.
3. **Provenance manifest** — the cohort table's build metadata, to save alongside the iteration.
4. **Column source-of-truth assessment** — pointer to the AI/Gemini step.

In [ ]:
# === Parameters — set per manuscript, then Run All ===
PROJECT               = 'thyroid-canonical-pub-2026'
COHORT_DATASET        = 'pub_canonical'
COHORT_TABLE          = 'manuscript_cohort_v1'        # the new / current iteration
PRIOR_COHORT_DATASET  = 'pub_archive'
PRIOR_COHORT_TABLE    = 'manuscript_cohort_v1'        # the previous iteration (often a pub_archive snapshot)

In [ ]:
from google.cloud import bigquery
import pandas as pd

client = bigquery.Client(project=PROJECT)
NEW   = f'`{PROJECT}.{COHORT_DATASET}.{COHORT_TABLE}`'
PRIOR = f'`{PROJECT}.{PRIOR_COHORT_DATASET}.{PRIOR_COHORT_TABLE}`'

def q(sql):
    return client.query(sql).to_dataframe()

print('new   :', NEW)
print('prior :', PRIOR)

## 1. Iteration diff

In [ ]:
# 1a. Patients added / dropped / unchanged between iterations
summary = q(f'''
WITH new_ids AS (SELECT DISTINCT research_id FROM {NEW}),
     old_ids AS (SELECT DISTINCT research_id FROM {PRIOR})
SELECT 'added'     AS change, COUNT(*) AS n FROM (SELECT research_id FROM new_ids EXCEPT DISTINCT SELECT research_id FROM old_ids)
UNION ALL SELECT 'dropped',   COUNT(*)        FROM (SELECT research_id FROM old_ids EXCEPT DISTINCT SELECT research_id FROM new_ids)
UNION ALL SELECT 'unchanged', COUNT(*)        FROM (SELECT research_id FROM new_ids INTERSECT DISTINCT SELECT research_id FROM old_ids)
''')
display(summary)

# the actual added / dropped research_ids — every add/drop needs a reason before locking
added_dropped = q(f'''
SELECT 'added' AS change, research_id
FROM (SELECT DISTINCT research_id FROM {NEW} EXCEPT DISTINCT SELECT DISTINCT research_id FROM {PRIOR})
UNION ALL
SELECT 'dropped', research_id
FROM (SELECT DISTINCT research_id FROM {PRIOR} EXCEPT DISTINCT SELECT DISTINCT research_id FROM {NEW})
ORDER BY change, research_id
''')
display(added_dropped)

In [ ]:
# 1b. Which numeric / boolean columns MOVED between iterations.
#     Compares non-null count and mean of every column shared by both iterations.
cols = q(f'''
SELECT n.column_name
FROM `{PROJECT}.{COHORT_DATASET}`.INFORMATION_SCHEMA.COLUMNS n
JOIN `{PROJECT}.{PRIOR_COHORT_DATASET}`.INFORMATION_SCHEMA.COLUMNS o
  ON n.column_name = o.column_name
WHERE n.table_name = '{COHORT_TABLE}' AND o.table_name = '{PRIOR_COHORT_TABLE}'
  AND n.data_type IN ('INT64','FLOAT64','NUMERIC','BIGNUMERIC','BOOL')
''')

if len(cols):
    parts = [
        f"SELECT '{c}' AS col, "
        f"(SELECT COUNT({c}) FROM {NEW}) AS n_new, (SELECT COUNT({c}) FROM {PRIOR}) AS n_prior, "
        f"(SELECT ROUND(AVG(CAST({c} AS FLOAT64)),4) FROM {NEW}) AS mean_new, "
        f"(SELECT ROUND(AVG(CAST({c} AS FLOAT64)),4) FROM {PRIOR}) AS mean_prior"
        for c in cols.column_name
    ]
    drift = q(' UNION ALL '.join(parts))
    drift['moved'] = (drift.n_new != drift.n_prior) | (drift.mean_new.fillna(-999) != drift.mean_prior.fillna(-999))
    moved = drift[drift.moved].sort_values('col')
    display(moved)
    print(f'{len(moved)} of {len(drift)} numeric/bool columns moved between iterations.')
    print('Every moved headline number needs an explained reason before this iteration is locked.')
else:
    print('No shared numeric/bool columns found between the two iterations.')

## 2. Cohort-scoped QC

The per-manuscript counterpart to the project-wide `cowork_qc_nonblocking_pipeline_v1`. Each check is guarded — a column not present in this cohort reports `skipped`, it does not crash. Full catalog: `pub_workspace.qc_rules_v1`.

In [ ]:
qc_checks = {
    'SURG01_surgery_date_divergence':    'DATE(first_surgery_date)<>DATE(surg_first_date) OR DATE(first_surgery_date)<>DATE(surgery_date) OR DATE(surg_first_date)<>DATE(surgery_date)',
    'LN01_ln_positive_gt_examined':      'ln_positive_final > path_ln_examined_raw',
    'LN02_ln_positive_without_examined': 'ln_positive_final > 0 AND COALESCE(path_ln_examined_raw,0)=0',
    'LN03_ln_raw_vs_final_disagree':     'path_ln_positive_raw IS NOT NULL AND ln_positive_final IS NOT NULL AND path_ln_positive_raw <> ln_positive_final',
    'REC02_recurrence_flag_without_date':'any_recurrence_flag = TRUE AND recurrence_date IS NULL',
    'REC03_recurrence_date_without_flag':'recurrence_date IS NOT NULL AND any_recurrence_flag IS NOT TRUE',
    'HIST01_histology_whitespace':       'histology_final <> TRIM(histology_final)',
}
rows = []
for rule, pred in qc_checks.items():
    try:
        n = int(client.query(f'SELECT COUNT(*) n FROM {NEW} WHERE {pred}').to_dataframe().n[0])
        rows.append({'rule': rule, 'violations': n, 'status': 'PASS' if n == 0 else 'FAIL'})
    except Exception:
        rows.append({'rule': rule, 'violations': None, 'status': 'skipped (column not in cohort)'})
qc = pd.DataFrame(rows)
display(qc)

fails = qc[qc.status == 'FAIL']
if len(fails):
    print(f'STOP-THE-LINE: {len(fails)} critical QC rule(s) failing on this cohort.')
    print('File/update a Linear issue (Database Reconciliation & QA) and do NOT lock numbers over a critical violation.')
else:
    print('Cohort-scoped QC: all checked rules pass.')

## 3. Provenance manifest

Save this alongside the iteration so a reviewer's “where did this N come from” is answerable later.

In [ ]:
import datetime
prov = q(f'''
SELECT table_id,
       TIMESTAMP_MILLIS(creation_time)       AS creation_time,
       TIMESTAMP_MILLIS(last_modified_time)  AS last_modified_time,
       row_count, size_bytes
FROM `{PROJECT}.{COHORT_DATASET}.__TABLES__`
WHERE table_id = '{COHORT_TABLE}'
''')
display(prov)

manifest = (
    f"# Provenance manifest — {COHORT_TABLE}\n"
    f"generated: {datetime.datetime.utcnow().isoformat()}Z\n"
    f"cohort: {PROJECT}.{COHORT_DATASET}.{COHORT_TABLE}\n"
    f"prior:  {PROJECT}.{PRIOR_COHORT_DATASET}.{PRIOR_COHORT_TABLE}\n"
    f"row_count: {int(prov.row_count[0]) if len(prov) else 'n/a'}\n"
    f"last_modified: {prov.last_modified_time[0] if len(prov) else 'n/a'}\n"
    f"diff vs prior: {summary.to_dict('records')}\n"
    f"qc: {qc[['rule','status']].to_dict('records')}\n"
)
print(manifest)
# Save next to the manuscript's evidence pack, e.g.:
# open(f'{COHORT_TABLE}_provenance.md','w').write(manifest)

## 4. Column source-of-truth assessment

Run `.cowork/skills/manuscript-iterative-build/sql/manuscript_column_source_assessment.sql` against this cohort (set its `cohort_table`). It flags every column the manuscript uses that is a competing source of truth — surgery date (THY-87), LN-positive (THY-89), histology, recurrence — and, where a Vertex AI connection is configured, adds a Gemini plain-language advisory. If the manuscript leans on a contested column, record it in the methods notes; do not silently bake in a column that is about to be deprecated.

**Division of labor:** this notebook + that SQL do detection and diffing every iteration. Picking the authoritative source for a contested column is a human decision (THY-87, THY-89) — do not resolve it autonomously inside a manuscript build.